# Grupo 12 — Reconhecimento de Gestos ASL em Tempo Real
Este notebook carrega o modelo treinado no `Grupo12_2_static_melhorado.ipynb` e
realiza reconhecimento de gestos ASL em tempo real através da webcam.

**Como usar:**
- Execute as células por ordem.
- A janela de vídeo abre automaticamente.
- Pressione **`q`** na janela de vídeo para fechar.
- Pressione **`c`** para limpar o texto acumulado.
- Mantenha a mão visível e estável durante pelo menos 5 frames para confirmar um gesto.

## 1. Importações e carregamento do modelo

In [4]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import os

# Verificar que os ficheiros do modelo existem antes de carregar
MODEL_PATHS = {
    "mlp":           "models/mlp_asl_landmarks.joblib",
    "scaler":        "models/scaler_asl_landmarks.joblib",
    "label_encoder": "models/label_encoder_asl_landmarks.joblib"
}
for name, path in MODEL_PATHS.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Ficheiro do modelo não encontrado: {path}\nCorre primeiro o Grupo12_2_static_melhorado.ipynb.")

mlp           = joblib.load(MODEL_PATHS["mlp"])
scaler        = joblib.load(MODEL_PATHS["scaler"])
label_encoder = joblib.load(MODEL_PATHS["label_encoder"])

print(f"Modelo carregado. Classes: {list(label_encoder.classes_)}")

# Configurar MediaPipe
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,       # False = modo vídeo (tracking entre frames)
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
print("MediaPipe configurado.")

Modelo carregado. Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'space']
MediaPipe configurado.


I0000 00:00:1779294314.053518   19796 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779294314.054293   19968 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)


## 2. Função de normalização de landmarks
Deve ser **idêntica** à usada no `Grupo12_1_static_melhorado.ipynb` para garantir
consistência entre extração e inferência.

In [5]:
def normalize_landmarks(lms):
    """
    Normaliza 21 landmarks de mão em 84 features:
      - 63 coordenadas relativas ao pulso (x, y, z por landmark)
      - 21 distâncias euclidianas ao pulso
    Retorna None se a escala for inválida (mão mal detetada).
    """
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z

    # Escala robusta: distância média do pulso às bases dos 4 dedos
    base_ids = [5, 9, 13, 17]
    dists_base = [
        ((lms[i].x - wx)**2 + (lms[i].y - wy)**2 + (lms[i].z - wz)**2) ** 0.5
        for i in base_ids
    ]
    scale = float(np.mean(dists_base))
    if scale < 1e-6:
        return None

    features = []

    # Coordenadas relativas (63 valores)
    for lm in lms:
        features.extend([
            (lm.x - wx) / scale,
            (lm.y - wy) / scale,
            (lm.z - wz) / scale
        ])

    # Distâncias euclidianas ao pulso (21 valores)
    for lm in lms:
        d = ((lm.x - wx)**2 + (lm.y - wy)**2 + (lm.z - wz)**2) ** 0.5
        features.append(d / scale)

    assert len(features) == 84, f"Erro interno: {len(features)} features (esperado 84)"
    return features

## 3. Loop principal — Reconhecimento em tempo real

In [6]:
# ── Parâmetros ───────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 5    # Nº de frames consecutivos para confirmar um gesto
CAMERA_INDEX = 0            # Índice da webcam (0 = câmara principal)
WINDOW_NAME = "Tradutor ASL — Grupo 12"

# ── Estado ───────────────────────────────────────────────────────────────────
current_letter     = ""
consecutive_frames = 0
confirmed_letter   = ""     # Última letra confirmada (exibida no ecrã)
text_buffer        = ""     # Texto acumulado das letras confirmadas

# ── Abrir webcam ─────────────────────────────────────────────────────────────
# Fecha janelas antigas caso a célula tenha sido interrompida antes
cv2.destroyAllWindows()

# Em Linux, CAP_V4L2 costuma ser mais estável para webcam com OpenCV
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_V4L2)

if not cap.isOpened():
    raise RuntimeError(f"Não foi possível abrir a câmara (índice {CAMERA_INDEX}).")

# Criar a janela UMA só vez, fora do ciclo while
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, 900, 700)

print("Webcam aberta. Pressione 'q' para sair, 'c' para limpar o texto.")

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("[AVISO] Frame não recebida — a terminar.")
            break

        # Espelhar para feedback natural e converter para RGB
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Processar com MediaPipe
        results = hands.process(rgb_frame)

        predicted_label = ""
        confidence_pct = 0.0

        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]  # Apenas 1 mão

            # Desenhar landmarks
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            features = normalize_landmarks(hand_landmarks.landmark)

            if features is not None:
                features_array = np.array(features).reshape(1, -1)

                # Evita o warning:
                # "X does not have valid feature names..."
                if hasattr(scaler, "feature_names_in_"):
                    import pandas as pd
                    features_array = pd.DataFrame(features_array, columns=scaler.feature_names_in_)

                features_scaled = scaler.transform(features_array)

                # Previsão + probabilidade
                prediction = mlp.predict(features_scaled)
                proba = mlp.predict_proba(features_scaled).max()
                predicted_label = label_encoder.inverse_transform(prediction)[0]
                confidence_pct = proba * 100

        # ── Lógica de suavização (smoothing) ─────────────────────────────────
        if predicted_label and predicted_label == current_letter:
            consecutive_frames += 1
        else:
            consecutive_frames = 1
            current_letter = predicted_label

        # Confirmar letra após CONFIDENCE_THRESHOLD frames consecutivos
        if consecutive_frames == CONFIDENCE_THRESHOLD and current_letter:
            confirmed_letter = current_letter

            if current_letter == "space":
                text_buffer += " "
            elif current_letter == "del":
                text_buffer = text_buffer[:-1]
            else:
                text_buffer += current_letter

        # ── HUD (heads-up display) ───────────────────────────────────────────
        h, w = frame.shape[:2]

        # Fundo semi-transparente no topo
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 80), (0, 0, 0), -1)
        frame = cv2.addWeighted(overlay, 0.45, frame, 0.55, 0)

        # Letra atual
        display_text = current_letter if consecutive_frames >= CONFIDENCE_THRESHOLD else "..."
        cv2.putText(
            frame,
            f"Gesto: {display_text}",
            (10, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.4,
            (0, 255, 0),
            3,
            cv2.LINE_AA
        )

        # Confiança do modelo
        if predicted_label:
            conf_color = (0, 255, 0) if confidence_pct >= 70 else (0, 200, 255)
            cv2.putText(
                frame,
                f"{confidence_pct:.0f}%",
                (w - 100, 45),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                conf_color,
                2,
                cv2.LINE_AA
            )

        # Barra de progresso de frames consecutivos
        bar_w = int((consecutive_frames / CONFIDENCE_THRESHOLD) * 200)
        bar_w = min(bar_w, 200)
        cv2.rectangle(frame, (10, 60), (10 + bar_w, 72), (0, 255, 0), -1)
        cv2.rectangle(frame, (10, 60), (210, 72), (100, 100, 100), 1)

        # Texto acumulado
        display_buffer = text_buffer[-30:] if len(text_buffer) > 30 else text_buffer
        cv2.putText(
            frame,
            f"Texto: {display_buffer}",
            (10, h - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        # Mostrar frame na mesma janela
        cv2.imshow(WINDOW_NAME, frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            print("A sair...")
            break
        elif key == ord("c"):
            text_buffer = ""
            print("[INFO] Texto limpo.")

finally:
    # ── Limpeza ─────────────────────────────────────────────────────────────
    cap.release()
    cv2.destroyAllWindows()
    hands.close()
    print(f"Sessão terminada. Texto final: '{text_buffer}'")


Webcam aberta. Pressione 'q' para sair, 'c' para limpar o texto.
Sessão terminada. Texto final: 'ABCDEFG HIOZESNNOCHHZHPPZZQRS TUURUVWUDZZUXDDZXXIYUUUX'


KeyboardInterrupt: 